In [2]:
# %%
import os
import sys
import numpy as np
import pandas as pd
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    raise RuntimeError("CUDA GPU is not available.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA: 12.8


In [3]:
# %%
print("Python:", sys.version)

print("\nKaggle input directories:")
for dirname, _, filenames in os.walk("/kaggle/input/datasets/faribaghorbani/autoformer-dataset"):
    level = dirname.replace("/kaggle/input", "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(dirname)}/")

    for filename in filenames:
        print(f"{indent}    {filename}")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Kaggle input directories:
            autoformer-dataset/
                dataset/
                    ETT-small/
                        ETTh2.csv
                        ETTm2.csv
                        ETTm1.csv
                        ETTh1.csv
                    exchange_rate/
                        exchange_rate.csv
                    illness/
                        national_illness.csv
                    weather/
                        weather.csv


In [4]:
# %%
import os

REPO = "/kaggle/working/Autoformer"

# Remove any previous repository from this Kaggle session
if os.path.exists(REPO):
    !rm -rf /kaggle/working/Autoformer

%cd /kaggle/working

!git clone --branch experiment1 --single-branch \
    https://github.com/faribaghorbani/Autoformer.git \
    Autoformer

/kaggle/working
Cloning into 'Autoformer'...
remote: Enumerating objects: 398, done.
remote: Counting objects: 100% (281/281), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 398 (delta 210), reused 196 (delta 187), pack-reused 117 (from 1)
Receiving objects: 100% (398/398), 2.22 MiB | 13.28 MiB/s, done.
Resolving deltas: 100% (236/236), done.


In [5]:
# %%
%cd /kaggle/working/Autoformer

!git status
!git branch --show-current
!git log -1 --oneline

/kaggle/working/Autoformer
On branch experiment1
Your branch is up to date with 'origin/experiment1'.

nothing to commit, working tree clean
experiment1
e851491 (HEAD -> experiment1, origin/experiment1) change run.py


In [6]:
# %%
branch = !git branch --show-current
branch = branch[0].strip()

commit = !git rev-parse HEAD
commit = commit[0].strip()

remote = !git remote get-url origin
remote = remote[0].strip()

print("Repository:", remote)
print("Branch:", branch)
print("Commit:", commit)

assert branch == "experiment1", (
    f"Wrong branch! Expected experiment1, got {branch}"
)

print("\n✓ Correct experiment branch loaded.")

Repository: https://github.com/faribaghorbani/Autoformer.git
Branch: experiment1
Commit: e851491bdb7fc06d6e73ac96886ab4c8717fb7d5

✓ Correct experiment branch loaded.


In [7]:
# %%
%cd /kaggle/working/Autoformer

print("Checking adaptive decomposition implementation...\n")

!grep -n "adaptive_multi_scale_series_decomp" layers/Autoformer_EncDec.py
!grep -n "decomp_type" layers/Autoformer_EncDec.py
!grep -n "decomp_kernels" layers/Autoformer_EncDec.py

/kaggle/working/Autoformer
Checking adaptive decomposition implementation...

52:class adaptive_multi_scale_series_decomp(nn.Module):
74:        super(adaptive_multi_scale_series_decomp, self).__init__()
256:        return adaptive_multi_scale_series_decomp(
225:    decomp_type,
235:        decomp_type:
252:    if decomp_type == "fixed":
255:    elif decomp_type == "adaptive":
263:            f"Unknown decomp_type='{decomp_type}'. "
280:        decomp_type="fixed",
307:            decomp_type=decomp_type,
314:            decomp_type=decomp_type,
387:        decomp_type="fixed",
415:            decomp_type=decomp_type,
422:            decomp_type=decomp_type,
429:            decomp_type=decomp_type,
228:    decomp_kernels=(13, 25, 49),
245:        decomp_kernels:
258:            kernel_sizes=decomp_kernels,
281:        decomp_kernels=(13, 25, 49),
310:            decomp_kernels=decomp_kernels,
317:            decomp_kernels=decomp_kernels,
388:        decomp_kernels=(13, 25, 49),
418:  

In [8]:
# %%
!grep -n "decomp_type" run.py
!grep -n "decomp_kernels" run.py

67:        '--decomp_type',
79:        '--decomp_kernels',


# Dependencies

In [9]:
# %%
%cd /kaggle/working/Autoformer

!pip install -q -r requirements.txt

/kaggle/working/Autoformer


In [10]:
# %%
import pandas
import numpy
import sklearn
import matplotlib
import reformer_pytorch

print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("sklearn:", sklearn.__version__)
print("matplotlib:", matplotlib.__version__)

print("\n✓ Dependencies imported successfully.")

pandas: 2.3.3
numpy: 2.0.2
sklearn: 1.6.1
matplotlib: 3.10.0

✓ Dependencies imported successfully.


# Dataset root

In [11]:
# %%
DATASET_ROOT = "/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset"

print("Dataset root:")
print(DATASET_ROOT)

assert os.path.exists(DATASET_ROOT), (
    f"Dataset root does not exist: {DATASET_ROOT}"
)

Dataset root:
/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset


In [12]:
# %%
for root, dirs, files in os.walk(DATASET_ROOT):
    level = root.replace(DATASET_ROOT, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

dataset/
    ETT-small/
        ETTh2.csv
        ETTm2.csv
        ETTm1.csv
        ETTh1.csv
    exchange_rate/
        exchange_rate.csv
    illness/
        national_illness.csv
    weather/
        weather.csv


In [13]:
# %%
ETTm2_ROOT = os.path.join(
    DATASET_ROOT,
    "ETT-small"
)

ETTm2_FILE = os.path.join(
    ETTm2_ROOT,
    "ETTm2.csv"
)

print("ETTm2 root:")
print(ETTm2_ROOT)

print("\nETTm2 file:")
print(ETTm2_FILE)

assert os.path.exists(ETTm2_FILE), (
    f"ETTm2.csv not found at {ETTm2_FILE}"
)

df = pd.read_csv(ETTm2_FILE)

print("\nShape:", df.shape)
print("Columns:", list(df.columns))
print("\nFirst rows:")
display(df.head())

ETTm2 root:
/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small

ETTm2 file:
/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small/ETTm2.csv

Shape: (69680, 8)
Columns: ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']

First rows:


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,41.130001,12.481,36.535999,9.355,4.424,1.311,38.661999
1,2016-07-01 00:15:00,39.622002,11.309,35.543999,8.551,3.209,1.258,38.223000
2,2016-07-01 00:30:00,38.868000,10.555,34.365002,7.586,4.435,1.258,37.344002
3,2016-07-01 00:45:00,35.518002,9.214,32.569000,8.712,4.435,1.215,37.124001
4,2016-07-01 01:00:00,37.528000,10.136,33.936001,7.532,4.435,1.215,37.124001


# experiment configuration

In [14]:
# %%
EXPERIMENT_NAME = "experiment1_adaptive_multiscale_decomposition"

DECOMP_TYPE = "adaptive"
DECOMP_KERNELS = [13, 25, 49]

SEQ_LEN = 96
LABEL_LEN = 48

PREDICTION_LENGTHS = [96, 192, 336, 720]

ENC_IN = 7
DEC_IN = 7
C_OUT = 7

E_LAYERS = 2
D_LAYERS = 1
D_MODEL = 512
FACTOR = 1

BATCH_SIZE = 32
LEARNING_RATE = 1e-4

TRAIN_EPOCHS = 10
PATIENCE = 3

SEED = 2021

print("Experiment:", EXPERIMENT_NAME)
print("Decomposition:", DECOMP_TYPE)
print("Kernels:", DECOMP_KERNELS)
print("Sequence length:", SEQ_LEN)
print("Prediction lengths:", PREDICTION_LENGTHS)

Experiment: experiment1_adaptive_multiscale_decomposition
Decomposition: adaptive
Kernels: [13, 25, 49]
Sequence length: 96
Prediction lengths: [96, 192, 336, 720]


## make the result file destination

In [15]:
# %%
RESULTS_FILE = (
    "/kaggle/working/"
    "autoformer_experiment1_results.csv"
)

columns = [
    "experiment",
    "git_branch",
    "git_commit",

    "dataset",
    "root_path",
    "data_path",

    "features",

    "seq_len",
    "label_len",
    "pred_len",

    "enc_in",
    "dec_in",
    "c_out",

    "e_layers",
    "d_layers",

    "factor",
    "d_model",

    "batch_size",
    "learning_rate",

    "train_epochs",
    "patience",

    "seed",

    "decomp_type",
    "decomp_kernels",

    "mse",
    "mae",

    "status",
    "return_code",

    "start_time",
]

if not os.path.exists(RESULTS_FILE):

    pd.DataFrame(columns=columns).to_csv(
        RESULTS_FILE,
        index=False
    )

print("Results file:")
print(RESULTS_FILE)

print("\nCurrent contents:")

display(
    pd.read_csv(RESULTS_FILE)
)

Results file:
/kaggle/working/autoformer_experiment1_results.csv

Current contents:


,experiment,git_branch,git_commit,dataset,root_path,data_path,features,seq_len,label_len,pred_len,...,train_epochs,patience,seed,decomp_type,decomp_kernels,mse,mae,status,return_code,start_time


## sanity check the new decomposition

In [16]:
# %%
%cd /kaggle/working/Autoformer

import torch

from layers.Autoformer_EncDec import (
    adaptive_multi_scale_series_decomp
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# Test with ETTm2 raw input channels
x = torch.randn(
    2,
    SEQ_LEN,
    ENC_IN,
    device=device
)

decomp = adaptive_multi_scale_series_decomp(
    channels=ENC_IN,
    kernel_sizes=DECOMP_KERNELS,
).to(device)

seasonal, trend = decomp(x)

print("Input shape:    ", x.shape)
print("Seasonal shape: ", seasonal.shape)
print("Trend shape:    ", trend.shape)

assert seasonal.shape == x.shape
assert trend.shape == x.shape

print("\n✓ Adaptive decomposition test passed.")

/kaggle/working/Autoformer
Device: cuda
Input shape:     torch.Size([2, 96, 7])
Seasonal shape:  torch.Size([2, 96, 7])
Trend shape:     torch.Size([2, 96, 7])

✓ Adaptive decomposition test passed.


In [17]:
# %%
x_hidden = torch.randn(
    2,
    SEQ_LEN,
    D_MODEL,
    device=device
)

decomp_hidden = adaptive_multi_scale_series_decomp(
    channels=D_MODEL,
    kernel_sizes=DECOMP_KERNELS,
).to(device)

seasonal_hidden, trend_hidden = decomp_hidden(x_hidden)

print("Input shape:    ", x_hidden.shape)
print("Seasonal shape: ", seasonal_hidden.shape)
print("Trend shape:    ", trend_hidden.shape)

assert seasonal_hidden.shape == x_hidden.shape
assert trend_hidden.shape == x_hidden.shape

print("\n✓ Hidden representation decomposition test passed.")

Input shape:     torch.Size([2, 96, 512])
Seasonal shape:  torch.Size([2, 96, 512])
Trend shape:     torch.Size([2, 96, 512])

✓ Hidden representation decomposition test passed.


## compile the modified files

In [18]:
# %%
%cd /kaggle/working/Autoformer

!python -m py_compile layers/Autoformer_EncDec.py
!python -m py_compile models/Autoformer.py
!python -m py_compile run.py

print("✓ All modified Python files compile successfully.")

/kaggle/working/Autoformer
✓ All modified Python files compile successfully.


In [19]:
# %%
%cd /kaggle/working/Autoformer

!python run.py --help | grep -A 5 -B 2 decomp

/kaggle/working/Autoformer
2026-08-13:07:28:22,027 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
              [--factor FACTOR] [--distil] [--dropout DROPOUT] [--embed EMBED]
              [--activation ACTIVATION] [--output_attention] [--do_predict]
              [--decomp_type {fixed,adaptive}]
              [--decomp_kernels DECOMP_KERNELS [DECOMP_KERNELS ...]]
              [--num_workers NUM_WORKERS] [--itr ITR]
              [--train_epochs TRAIN_EPOCHS] [--batch_size BATCH_SIZE]
              [--patience PATIENCE] [--learning_rate LEARNING_RATE]
              [--des DES] [--loss LOSS] [--lradj LRADJ] [--use_amp]
              [--use_gpu USE_GPU] [--gpu GPU] [--use_multi_gpu]
--
  --output_attention    whether to output attention in encoder
  --do_predict          whether to predict unseen future data
  --decomp_type {fixed,adaptive}
                        Type of series decomposition: fixed uses the original
                        Autoformer moving average; adaptiv

## Design the experiment runner

In [20]:
# %%
import os
import re
import subprocess
import pandas as pd

from datetime import datetime


REPO = "/kaggle/working/Autoformer"

RESULTS_FILE = (
    "/kaggle/working/"
    "autoformer_experiment1_results.csv"
)


def run_autoformer_experiment(
    dataset,
    root_path,
    data_path,
    data_type,
    seq_len,
    label_len,
    pred_len,
    enc_in,
    dec_in,
    c_out,
    freq,

    train_epochs=10,
    patience=3,
    factor=1,
    batch_size=32,
    learning_rate=1e-4,
    seed=2021,

    decomp_type="adaptive",
    decomp_kernels=(13, 25, 49),
):
    """
    Run one Autoformer experiment and append its result
    to the experiment1 results CSV.
    """

    # ---------------------------------------------------------
    # Experiment metadata
    # ---------------------------------------------------------

    branch = subprocess.check_output(
        ["git", "branch", "--show-current"],
        cwd=REPO,
        text=True
    ).strip()

    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=REPO,
        text=True
    ).strip()

    # ---------------------------------------------------------
    # Model ID
    # ---------------------------------------------------------

    model_id = (
        f"{dataset}_"
        f"{seq_len}_"
        f"{pred_len}_"
        f"experiment1"
    )

    # ---------------------------------------------------------
    # Convert kernels to strings for command line
    # ---------------------------------------------------------

    decomp_kernels = tuple(
        int(k)
        for k in decomp_kernels
    )

    kernel_args = [
        str(k)
        for k in decomp_kernels
    ]

    # ---------------------------------------------------------
    # Build command
    # ---------------------------------------------------------

    cmd = [
        "python", "-u", "run.py",

        "--is_training", "1",

        "--root_path", root_path,
        "--data_path", data_path,

        "--model_id", model_id,
        "--model", "Autoformer",
        "--data", data_type,

        "--features", "M",

        "--seq_len", str(seq_len),
        "--label_len", str(label_len),
        "--pred_len", str(pred_len),

        "--e_layers", str(E_LAYERS),
        "--d_layers", str(D_LAYERS),

        "--factor", str(factor),

        "--enc_in", str(enc_in),
        "--dec_in", str(dec_in),
        "--c_out", str(c_out),

        "--d_model", str(D_MODEL),

        "--batch_size", str(batch_size),
        "--learning_rate", str(learning_rate),

        "--train_epochs", str(train_epochs),
        "--patience", str(patience),

        "--des", "experiment1",

        # -----------------------------------------------------
        # NEW ARCHITECTURE
        # -----------------------------------------------------

        "--decomp_type", str(decomp_type),

        "--decomp_kernels",
        *kernel_args,

        "--itr", "1",
    ]

    # ---------------------------------------------------------
    # Print experiment information
    # ---------------------------------------------------------

    print("=" * 100)
    print("EXPERIMENT 1")
    print("=" * 100)

    print(f"Dataset:          {dataset}")
    print(f"Input length:     {seq_len}")
    print(f"Prediction:       {pred_len}")

    print(f"Decomposition:    {decomp_type}")
    print(f"Decomp kernels:   {decomp_kernels}")

    print(f"Git branch:       {branch}")
    print(f"Git commit:       {commit}")

    print("=" * 100)

    print("\nCommand:")
    print(" ".join(cmd))

    print("\n" + "=" * 100)
    print("LIVE TRAINING LOG")
    print("=" * 100)

    start = datetime.now()

    # ---------------------------------------------------------
    # Run process
    # ---------------------------------------------------------

    process = subprocess.Popen(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    output_lines = []

    for line in iter(
        process.stdout.readline,
        ""
    ):
        print(
            line,
            end="",
            flush=True
        )

        output_lines.append(line)

    process.stdout.close()

    return_code = process.wait()

    # ---------------------------------------------------------
    # Combine output
    # ---------------------------------------------------------

    output = "".join(output_lines)

    # ---------------------------------------------------------
    # Extract final MSE and MAE
    # ---------------------------------------------------------

    matches = re.findall(
        r"mse:([0-9eE.+-]+), mae:([0-9eE.+-]+)",
        output
    )

    mse = None
    mae = None

    if matches:

        mse = float(
            matches[-1][0]
        )

        mae = float(
            matches[-1][1]
        )

    # ---------------------------------------------------------
    # Status
    # ---------------------------------------------------------

    status = (
        "success"
        if (
            return_code == 0
            and mse is not None
            and mae is not None
        )
        else "failed"
    )

    # ---------------------------------------------------------
    # Result row
    # ---------------------------------------------------------

    row = {
        "experiment":
            EXPERIMENT_NAME,

        "git_branch":
            branch,

        "git_commit":
            commit,

        "dataset":
            dataset,

        "root_path":
            root_path,

        "data_path":
            data_path,

        "features":
            "M",

        "seq_len":
            seq_len,

        "label_len":
            label_len,

        "pred_len":
            pred_len,

        "enc_in":
            enc_in,

        "dec_in":
            dec_in,

        "c_out":
            c_out,

        "e_layers":
            E_LAYERS,

        "d_layers":
            D_LAYERS,

        "factor":
            factor,

        "d_model":
            D_MODEL,

        "batch_size":
            batch_size,

        "learning_rate":
            learning_rate,

        "train_epochs":
            train_epochs,

        "patience":
            patience,

        "seed":
            seed,

        "decomp_type":
            decomp_type,

        "decomp_kernels":
            ",".join(
                map(str, decomp_kernels)
            ),

        "mse":
            mse,

        "mae":
            mae,

        "status":
            status,

        "return_code":
            return_code,

        "start_time":
            str(start),
    }

    result_df = pd.DataFrame(
        [row]
    )

    # ---------------------------------------------------------
    # Save
    # ---------------------------------------------------------

    result_df.to_csv(
        RESULTS_FILE,
        mode="a",
        header=False,
        index=False,
    )

    # ---------------------------------------------------------
    # Print summary
    # ---------------------------------------------------------

    print("\n" + "=" * 100)
    print("EXPERIMENT FINISHED")
    print("=" * 100)

    print(f"Return code: {return_code}")
    print(f"MSE:         {mse}")
    print(f"MAE:         {mae}")
    print(f"Status:      {status}")

    print("\nResult appended to:")
    print(RESULTS_FILE)

    display(result_df)

    return result_df

In [21]:
# %%
print("Repository:", REPO)
print("Results:", RESULTS_FILE)

print("\nArchitecture:")
print("  decomposition:", DECOMP_TYPE)
print("  kernels:", DECOMP_KERNELS)

print("\nModel:")
print("  d_model:", D_MODEL)
print("  encoder layers:", E_LAYERS)
print("  decoder layers:", D_LAYERS)

print("\nForecasting:")
print("  seq_len:", SEQ_LEN)
print("  label_len:", LABEL_LEN)
print("  prediction lengths:", PREDICTION_LENGTHS)

Repository: /kaggle/working/Autoformer
Results: /kaggle/working/autoformer_experiment1_results.csv

Architecture:
  decomposition: adaptive
  kernels: [13, 25, 49]

Model:
  d_model: 512
  encoder layers: 2
  decoder layers: 1

Forecasting:
  seq_len: 96
  label_len: 48
  prediction lengths: [96, 192, 336, 720]


# First run: horizon 96

In [22]:
# %%
result = run_autoformer_experiment(

    dataset="ETTm2",

    root_path=ETTm2_ROOT,
    data_path="ETTm2.csv",

    data_type="ETTm2",

    seq_len=96,
    label_len=48,
    pred_len=96,

    enc_in=7,
    dec_in=7,
    c_out=7,

    freq="t",

    train_epochs=10,
    patience=3,

    factor=1,

    batch_size=32,
    learning_rate=1e-4,

    seed=2021,

    decomp_type="adaptive",
    decomp_kernels=(13, 25, 49),
)

EXPERIMENT 1
Dataset:          ETTm2
Input length:     96
Prediction:       96
Decomposition:    adaptive
Decomp kernels:   (13, 25, 49)
Git branch:       experiment1
Git commit:       e851491bdb7fc06d6e73ac96886ab4c8717fb7d5

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_96_experiment1 --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 96 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --des experiment1 --decomp_type adaptive --decomp_kernels 13 25 49 --itr 1

LIVE TRAINING LOG
2026-08-13:07:31:39,282 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_96_experiment1', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoform

,experiment,git_branch,git_commit,dataset,root_path,data_path,features,seq_len,label_len,pred_len,...,train_epochs,patience,seed,decomp_type,decomp_kernels,mse,mae,status,return_code,start_time
0,experiment1_adaptive_multiscale_decomposition,experiment1,e851491bdb7fc06d6e73ac96886ab4c8717fb7d5,ETTm2,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,96,...,10,3,2021,adaptive,"13,25,49",0.232558,0.310838,success,0,2026-08-13 07:31:37.436132


## verify the correct architecture is used

In [24]:
# %%
results = pd.read_csv(
    RESULTS_FILE
)

display(results)

print(
    results[
        [
            "experiment",
            "git_branch",
            "git_commit",
            "decomp_type",
            "decomp_kernels",
            "pred_len",
            "mse",
            "mae",
            "status",
        ]
    ].tail()
)

,experiment,git_branch,git_commit,dataset,root_path,data_path,features,seq_len,label_len,pred_len,...,train_epochs,patience,seed,decomp_type,decomp_kernels,mse,mae,status,return_code,start_time
0,experiment1_adaptive_multiscale_decomposition,experiment1,e851491bdb7fc06d6e73ac96886ab4c8717fb7d5,ETTm2,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,96,...,10,3,2021,adaptive,"13,25,49",0.232558,0.310838,success,0,2026-08-13 07:31:37.436132


                                      experiment   git_branch  \
0  experiment1_adaptive_multiscale_decomposition  experiment1   

                                 git_commit decomp_type decomp_kernels  \
0  e851491bdb7fc06d6e73ac96886ab4c8717fb7d5    adaptive       13,25,49   

   pred_len       mse       mae   status  
0        96  0.232558  0.310838  success  


## Full experiments

In [25]:
# %%
ETTm2_ROOT = os.path.join(
    DATASET_ROOT,
    "ETT-small"
)

for pred_len in PREDICTION_LENGTHS:

    run_autoformer_experiment(

        dataset="ETTm2",

        root_path=ETTm2_ROOT,
        data_path="ETTm2.csv",

        data_type="ETTm2",

        seq_len=SEQ_LEN,
        label_len=LABEL_LEN,
        pred_len=pred_len,

        enc_in=ENC_IN,
        dec_in=DEC_IN,
        c_out=C_OUT,

        freq="t",

        train_epochs=TRAIN_EPOCHS,
        patience=PATIENCE,

        factor=FACTOR,

        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,

        seed=SEED,

        decomp_type=DECOMP_TYPE,
        decomp_kernels=DECOMP_KERNELS,
    )

EXPERIMENT 1
Dataset:          ETTm2
Input length:     96
Prediction:       96
Decomposition:    adaptive
Decomp kernels:   (13, 25, 49)
Git branch:       experiment1
Git commit:       e851491bdb7fc06d6e73ac96886ab4c8717fb7d5

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_96_experiment1 --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 96 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --des experiment1 --decomp_type adaptive --decomp_kernels 13 25 49 --itr 1

LIVE TRAINING LOG
2026-08-13:07:53:32,108 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_96_experiment1', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoform

,experiment,git_branch,git_commit,dataset,root_path,data_path,features,seq_len,label_len,pred_len,...,train_epochs,patience,seed,decomp_type,decomp_kernels,mse,mae,status,return_code,start_time
0,experiment1_adaptive_multiscale_decomposition,experiment1,e851491bdb7fc06d6e73ac96886ab4c8717fb7d5,ETTm2,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,96,...,10,3,2021,adaptive,"13,25,49",0.239337,0.316783,success,0,2026-08-13 07:53:30.226857


EXPERIMENT 1
Dataset:          ETTm2
Input length:     96
Prediction:       192
Decomposition:    adaptive
Decomp kernels:   (13, 25, 49)
Git branch:       experiment1
Git commit:       e851491bdb7fc06d6e73ac96886ab4c8717fb7d5

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_192_experiment1 --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 192 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --des experiment1 --decomp_type adaptive --decomp_kernels 13 25 49 --itr 1

LIVE TRAINING LOG
2026-08-13:08:10:25,465 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_192_experiment1', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/auto

,experiment,git_branch,git_commit,dataset,root_path,data_path,features,seq_len,label_len,pred_len,...,train_epochs,patience,seed,decomp_type,decomp_kernels,mse,mae,status,return_code,start_time
0,experiment1_adaptive_multiscale_decomposition,experiment1,e851491bdb7fc06d6e73ac96886ab4c8717fb7d5,ETTm2,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,192,...,10,3,2021,adaptive,"13,25,49",0.283323,0.345154,success,0,2026-08-13 08:10:23.554435


EXPERIMENT 1
Dataset:          ETTm2
Input length:     96
Prediction:       336
Decomposition:    adaptive
Decomp kernels:   (13, 25, 49)
Git branch:       experiment1
Git commit:       e851491bdb7fc06d6e73ac96886ab4c8717fb7d5

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_336_experiment1 --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 336 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --des experiment1 --decomp_type adaptive --decomp_kernels 13 25 49 --itr 1

LIVE TRAINING LOG
2026-08-13:08:43:44,377 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_336_experiment1', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/auto

,experiment,git_branch,git_commit,dataset,root_path,data_path,features,seq_len,label_len,pred_len,...,train_epochs,patience,seed,decomp_type,decomp_kernels,mse,mae,status,return_code,start_time
0,experiment1_adaptive_multiscale_decomposition,experiment1,e851491bdb7fc06d6e73ac96886ab4c8717fb7d5,ETTm2,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,336,...,10,3,2021,adaptive,"13,25,49",0.404706,0.406209,success,0,2026-08-13 08:43:42.544979


EXPERIMENT 1
Dataset:          ETTm2
Input length:     96
Prediction:       720
Decomposition:    adaptive
Decomp kernels:   (13, 25, 49)
Git branch:       experiment1
Git commit:       e851491bdb7fc06d6e73ac96886ab4c8717fb7d5

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_720_experiment1 --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 720 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --des experiment1 --decomp_type adaptive --decomp_kernels 13 25 49 --itr 1

LIVE TRAINING LOG
2026-08-13:09:14:43,882 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_720_experiment1', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/auto

,experiment,git_branch,git_commit,dataset,root_path,data_path,features,seq_len,label_len,pred_len,...,train_epochs,patience,seed,decomp_type,decomp_kernels,mse,mae,status,return_code,start_time
0,experiment1_adaptive_multiscale_decomposition,experiment1,e851491bdb7fc06d6e73ac96886ab4c8717fb7d5,ETTm2,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,720,...,10,3,2021,adaptive,"13,25,49",0.476838,0.4504,success,0,2026-08-13 09:14:42.053644
